<a href="https://colab.research.google.com/github/Castlebin/d2l-zh-pytorch-colab/blob/my_master/9_d2l-zh-pytorch-colab-reorg/04_%E5%A4%9A%E5%B1%82%E6%84%9F%E7%9F%A5%E6%9C%BA/04_%E5%88%86%E5%B8%83%E5%81%8F%E7%A7%BB%E4%B8%8E%E9%83%A8%E7%BD%B2%E6%8C%91%E6%88%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 分布偏移与部署挑战

本笔记改写自原 `environment.ipynb`,讨论机器学习模型在真实环境中面临的分布偏移(distribution shift)问题,以及如何监控与缓解。

## 1. 什么是分布偏移?

- 训练数据与部署环境的数据分布不一致。
- 可能导致模型在离线评估中表现良好,上线后性能骤降。

常见情形:

1. **协变量偏移(Covariate Shift)**: 输入特征分布变化,标签条件分布基本不变。
2. **先验偏移(Prior Shift)**: 标签分布变化,如欺诈率上升。
3. **概念漂移(Concept Drift)**: 特征与标签的关系改变,模型需要重新学习。

## 2. 真实案例

- 推荐系统面对季节性需求变化。
- 计算机视觉模型部署到不同光照/设备环境。
- NLP 情感分析模型遇到新潮流词汇或讽刺语气。

## 3. 小实验: 协变量偏移的影响

我们构造一个简单示例展示同一模型在两个分布下的表现差异。



In [1]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

# 训练集: 均值在 (0, 0)
x_train = torch.randn(1000, 2)
w_true = torch.tensor([[2.0], [-3.0]])
y_train = (x_train @ w_true + 0.5 > 0).float().squeeze()

# 测试集A: 分布相同
x_test_a = torch.randn(400, 2)
y_test_a = (x_test_a @ w_true + 0.5 > 0).float().squeeze()

# 测试集B: 协变量偏移(整体偏移 + 方差增大)
x_test_b = torch.randn(400, 2) * 1.5 + torch.tensor([1.0, -1.0])
y_test_b = (x_test_b @ w_true + 0.5 > 0).float().squeeze()

model = nn.Sequential(nn.Linear(2, 1))
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

data_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)

for epoch in range(20):
    for x_batch, y_batch in data_loader:
        logits = model(x_batch).squeeze()
        loss = loss_fn(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

sigmoid = nn.Sigmoid()

def accuracy(x, y):
    with torch.no_grad():
        preds = sigmoid(model(x)).squeeze() > 0.5
    return (preds == y.bool()).float().mean().item()

print(f'训练集准确率: {accuracy(x_train, y_train):.3f}')
print(f'测试集A准确率(分布一致): {accuracy(x_test_a, y_test_a):.3f}')
print(f'测试集B准确率(协变量偏移): {accuracy(x_test_b, y_test_b):.3f}')



训练集准确率: 0.999
测试集A准确率(分布一致): 0.998
测试集B准确率(协变量偏移): 1.000


可见测试集B的准确率显著下降,说明模型对分布变化非常敏感。

## 4. 分布偏移的监控

1. **特征统计监控**: 比较线上、离线的数据统计(均值、方差、最大/最小等)。
2. **模型输出监控**: 观察预测概率分布、置信度。
3. **延迟标签**: 设计机制持续收集真值,用于再训练或告警。
4. **漂移检测算法**: 如 PSI(Population Stability Index)、KL 散度、ADWIN。

## 5. 缓解策略

| 策略 | 说明 |
|------|------|
| 数据增强 | 模拟不同场景(亮度、噪声、语言风格) |
| 领域自适应 | 使用对抗训练或微调将源域映射到目标域 |
| 在线学习 | 持续更新模型参数,适应新数据 |
| 模型集成 | 组合多个在不同分布上训练的模型 |
| 置信度阈值 | 对低置信度预测进行人工审核 |

## 6. 部署 checklist

- [ ] 训练、验证、测试数据来自不同时间窗口
- [ ] 评估指标包含精准率、召回率、AUC 等多维度
- [ ] 建立自动化数据质量监控
- [ ] 预留回滚方案与蓝绿部署能力
- [ ] 定期重新训练,并对比新旧模型性能

## 7. 延伸阅读

- *Reliable Machine Learning: Applying SRE Principles to ML in Production*, Google SRE 团队
- Koh et al., 2020. *Understanding Black-box Predictions via Influence Functions*
- Lipton, 2018. *Detecting and Correcting for Label Shift with Black Box Predictors*

---
通过本笔记你已了解模型上线面临的常见风险。建议结合 `03_Kaggle实战_房价预测.ipynb` 的数据划分策略,设计更可靠的评估流程。

